In [54]:
import sys
print(sys.executable)
import statsmodels.api as sm 
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

df_apps = pd.read_csv(r"C:\Users\15612\OneDrive\Documents\googleplaystore.csv")
print(df_apps.head())


c:\Users\15612\OneDrive\Desktop\.venv\Scripts\python.exe
                                                 App        Category  Rating  \
0     Photo Editor & Candy Camera & Grid & ScrapBook  ART_AND_DESIGN     4.1   
1                                Coloring book moana  ART_AND_DESIGN     3.9   
2  U Launcher Lite – FREE Live Cool Themes, Hide ...  ART_AND_DESIGN     4.7   
3                              Sketch - Draw & Paint  ART_AND_DESIGN     4.5   
4              Pixel Draw - Number Art Coloring Book  ART_AND_DESIGN     4.3   

  Reviews  Size     Installs  Type Price Content Rating  \
0     159   19M      10,000+  Free     0       Everyone   
1     967   14M     500,000+  Free     0       Everyone   
2   87510  8.7M   5,000,000+  Free     0       Everyone   
3  215644   25M  50,000,000+  Free     0           Teen   
4     967  2.8M     100,000+  Free     0       Everyone   

                      Genres Last Updated         Current Ver   Android Ver  
0               Art & Design 

First step is cleaning our data and converting to acceptable format for analysis.

In [61]:
df_apps = df_apps.iloc[:-1].copy()
df_apps['Installs_clean'] = df_apps['Installs'].str.replace(r'[+,.]','', regex=True)
df_apps['Installs_clean'] = pd.to_numeric(df_apps['Installs_clean'], errors='coerce')

df_apps['Price_clean'] = df_apps['Price'].str.replace(r'[$,]', '',regex=True)
df_apps['Price_clean'] = pd.to_numeric(df_apps['Price_clean'], errors='coerce')

df_apps = df_apps.sort_values('Reviews', key=lambda s: pd.to_numeric(s, errors='coerce'), ascending=False) \
                 .drop_duplicates(subset='App', keep='first')

print(df_apps[['Installs_clean', 'Price_clean', 'Rating', 'Reviews']].dtypes)

Installs_clean      int64
Price_clean       float64
Rating            float64
Reviews               str
dtype: object


Exploring relationships between variables. I will start with continuous variables price and rating to determine the log of installs.  We found 15 instances of installs being 0, so we applied log (1+installs) for modeling purposes.

In [56]:
df_apps['log_Installs'] = np.log1p(df_apps['Installs_clean'])
model_1 = smf.ols('log_Installs ~ Rating + Price_clean', data=df_apps).fit()
print(model_1.summary())

                            OLS Regression Results                            
Dep. Variable:           log_Installs   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     42.56
Date:                Fri, 09 Oct 2026   Prob (F-statistic):           4.08e-19
Time:                        14:38:30   Log-Likelihood:                -22291.
No. Observations:                8195   AIC:                         4.459e+04
Df Residuals:                    8192   BIC:                         4.461e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       9.4118      0.319     29.543      

These results in coefficients are not the exact percentage change associated with raw installs yet. 

In [57]:
df_apps[['log_Installs','Rating', 'Price_clean']].corr()

,log_Installs,Rating,Price_clean
log_Installs,1.000000,0.084674,-0.056120
Rating,0.084674,1.000000,-0.021104
Price_clean,-0.056120,-0.021104,1.000000


In the following block we will be peforming group aggregation for the different types of app categories listed in our dataset and creating additional columns that will describe different statistics for each group. 

In [ ]:
category_summary = (
    df_apps.groupby('Category')
    .agg(
        app_count=('App', 'count'),
        mean_log_installs=('log_Installs', 'mean'),
        median_installs=('Installs_clean', 'median'),
        median_rating=('Rating', 'median'),
        rated_n=('Rating', 'count'),
        paid_share=('Price_clean', lambda x: (x > 0).mean()),
    )
    .sort_values('mean_log_installs', ascending=False)
)

print(category_summary.round(2))


                     app_count  mean_log_installs  median_installs  \
Category                                                             
ENTERTAINMENT               87              14.61        1000000.0   
EDUCATION                  106              13.13        1000000.0   
GAME                       945              13.03        1000000.0   
PHOTOGRAPHY                281              13.03        1000000.0   
WEATHER                     79              12.68        1000000.0   
VIDEO_PLAYERS              164              12.33        1000000.0   
SHOPPING                   202              12.28        1000000.0   
HOUSE_AND_HOME              73              11.77         500000.0   
FOOD_AND_DRINK             112              11.42         300000.0   
COMICS                      56              11.38         100000.0   
SOCIAL                     239              11.23         100000.0   
TRAVEL_AND_LOCAL           219              11.18         100000.0   
COMMUNICATION       

Comparing base model with 'Category' added model and adjusted R squared values.

In [ ]:
model_2 = df_apps.dropna(subset=['log_Installs', 'Rating', 'Price_clean', 'Category'])

m_base = smf.ols('log_Installs ~ Rating + Price_clean', data=model_2).fit()
m_cat  = smf.ols("log_Installs ~ Rating + Price_clean + C(Category, Treatment(reference='FAMILY'))",
                 data=model_2).fit()

print(m_base.rsquared_adj, m_cat.rsquared_adj)
print(sm.stats.anova_lm(m_base, m_cat))   


0.010042620980587058 0.10222882937308486
   df_resid            ssr  df_diff      ss_diff          F         Pr(>F)
0    8192.0  110581.503035      0.0          NaN        NaN            NaN
1    8160.0   99892.265255     32.0  10689.23778  27.286954  5.196479e-154


In [66]:
cat_effects = m_cat.params.filter(like='C(Category').sort_values()
(np.exp(cat_effects) - 1).mul(100).round(0)  

C(Category, Treatment(reference='FAMILY'))[T.MEDICAL]                 -92.0
C(Category, Treatment(reference='FAMILY'))[T.EVENTS]                  -86.0
C(Category, Treatment(reference='FAMILY'))[T.BUSINESS]                -72.0
C(Category, Treatment(reference='FAMILY'))[T.LIFESTYLE]               -44.0
C(Category, Treatment(reference='FAMILY'))[T.LIBRARIES_AND_DEMO]      -38.0
C(Category, Treatment(reference='FAMILY'))[T.AUTO_AND_VEHICLES]       -25.0
C(Category, Treatment(reference='FAMILY'))[T.DATING]                  -25.0
C(Category, Treatment(reference='FAMILY'))[T.FINANCE]                 -23.0
C(Category, Treatment(reference='FAMILY'))[T.PERSONALIZATION]         -21.0
C(Category, Treatment(reference='FAMILY'))[T.BEAUTY]                  -19.0
C(Category, Treatment(reference='FAMILY'))[T.BOOKS_AND_REFERENCE]      -4.0
C(Category, Treatment(reference='FAMILY'))[T.ART_AND_DESIGN]           -3.0
C(Category, Treatment(reference='FAMILY'))[T.PARENTING]                 1.0
C(Category, 

Leaving out Price will give us a higher adjusted R squared value. 

In [64]:
m_int = smf.ols("log_Installs ~ Rating + C(Type) * C(Category, Treatment(reference='FAMILY'))",
                data=model_2).fit()
print(m_int.rsquared_adj)
print(sm.stats.anova_lm(m_cat, m_int))


0.1699369274282353
   df_resid           ssr  df_diff      ss_diff          F         Pr(>F)
0    8160.0  99892.265255      0.0          NaN        NaN            NaN
1    8133.0  92052.993014     27.0  7839.272241  25.652213  5.691148e-123


C:\Users\15612\AppData\Local\Temp\ipykernel_33556\3485445322.py:2: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  data=model_2).fit()


Comparing models' results

In [67]:
models = {
    'base: Rating + Price':      'log_Installs ~ Rating + Price_clean',
    '+ Category':                "log_Installs ~ Rating + Price_clean + C(Category)",
    'swap Price -> Type':        "log_Installs ~ Rating + C(Type) + C(Category)",
    '+ Type x Category':         "log_Installs ~ Rating + C(Type) * C(Category)",
}
fits = {name: smf.ols(f, data=model_2).fit() for name, f in models.items()}
print(pd.Series({n: m.rsquared_adj for n, m in fits.items()}).round(3))

print(sm.stats.anova_lm(fits['swap Price -> Type'], fits['+ Type x Category']))


base: Rating + Price    0.010
+ Category              0.102
swap Price -> Type      0.167
+ Type x Category       0.170
dtype: float64
   df_resid           ssr  df_diff     ss_diff         F   Pr(>F)
0    8160.0  92731.355852      0.0         NaN       NaN      NaN
1    8133.0  92052.993014     27.0  678.362838  2.219786  0.00028


C:\Users\15612\AppData\Local\Temp\ipykernel_33556\3482612416.py:7: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fits = {name: smf.ols(f, data=model_2).fit() for name, f in models.items()}
